# 06. Genomic prediction

This notebook derives drought-response phenotypes for the 176-accession Brazil panel and tests whether context-specific GSL genetic scores predict grain-yield plasticity. It produces Figure 5, Table S5, and Dataset S5.


# 06. 基因组预测

本 notebook 为包含 176 个材料的 Brazil panel 构建 drought-response phenotypes，并检验 context-specific GSL genetic scores 是否预测 grain-yield plasticity，最终生成 Figure 5、Table S5 和 Dataset S5。


## Remote kernel initialization

The notebook runs against the remote project configuration. Verify the reported host, working directory, Python executable, environment, and expected kernelspec before continuing. This cell does not download data or start an analysis.


## 远程 kernel 初始化

本 notebook 根据远程项目配置运行。继续之前，请核对输出的 host、工作目录、Python executable、environment 和预期 kernelspec。本单元不会下载数据或启动分析。


In [ ]:
import os
import socket
import sys
from pathlib import Path

import yaml

%load_ext autoreload
%autoreload 2

launch_dir = Path.cwd().resolve()
project_file = launch_dir / "config" / "project.yaml"
if not project_file.is_file() and launch_dir.name == "notebooks":
    project_file = launch_dir.parent / "config" / "project.yaml"
if not project_file.is_file():
    raise FileNotFoundError("Copy config/project.example.yaml to config/project.yaml and complete it before running this notebook")
with project_file.open("r", encoding="utf-8") as stream:
    project_settings = yaml.safe_load(stream)
PROJECT_ROOT = Path(project_settings["remote"]["project_root"]).expanduser().resolve()
allowed_launch_dirs = {PROJECT_ROOT, PROJECT_ROOT / "notebooks"}
assert launch_dir in allowed_launch_dirs, f"Start Jupyter from the configured remote project root: {PROJECT_ROOT}"
assert PROJECT_ROOT.is_dir(), f"Remote project does not exist: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)
src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
expected_environment = project_settings["jupyter"]["kernel"]["environment"]["name"]
active_environment = os.environ.get("CONDA_DEFAULT_ENV", Path(sys.prefix).name)
if project_settings["jupyter"]["same_environment"]:
    server_environment = project_settings["jupyter"]["server"]["environment"]["name"]
    assert server_environment == expected_environment, (server_environment, expected_environment)
assert active_environment == expected_environment, (active_environment, expected_environment)
print({
    "host": socket.gethostname(),
    "working_directory": str(Path.cwd()),
    "python_executable": sys.executable,
    "environment": active_environment,
    "expected_kernelspec": project_settings["jupyter"]["kernel"]["kernelspec"],
})


## Run mode selection

Use `checkpoint` to read the published intermediate inputs or `full` to rebuild this notebook's intermediate products from source data. Outputs from the current run are always written to generated directories and never overwrite reference files.


## 运行模式选择

使用 `checkpoint` 读取已发布的中间输入；使用 `full` 从 source data 重建本 notebook 的中间产品。本次运行的输出始终写入 generated 目录，不覆盖 reference 文件。


In [ ]:
RUN_MODE = "checkpoint"
if RUN_MODE not in {"checkpoint", "full"}:
    raise ValueError("RUN_MODE must be checkpoint or full")
RUN_MODE


## Inputs and output contracts

All scientific inputs are resolved below `PROJECT_ROOT/data`. Published inputs are read from source or reference directories; newly built checkpoints and results are written only to generated directories. The following sections validate the notebook-specific schemas before interpreting results.


## 输入与输出契约

所有科学输入均从 `PROJECT_ROOT/data` 下解析。已发布输入从 source 或 reference 目录读取；新建 checkpoints 和 results 只写入 generated 目录。以下章节在解释结果前检查本 notebook 特有的 schemas。


## Scientific settings used in this notebook

The expected GxE accession count and random seed are declared with the prediction analysis. Trait derivation parameters are not required because averages and plasticity are supplied as analysis-ready source variables.


## 本 notebook 使用的科学设置

预期 GxE 样本数和随机种子在 prediction analysis 中直接声明。由于 averages 和 plasticity 已作为 analysis-ready source variables 提供，这里不需要 trait derivation parameters。


In [ ]:
RANDOM_SEED = 5196
EXPECTED_GXE_ACCESSION_COUNT = 176


In [ ]:
import sys
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
from laenrich.config import load_project_paths
from laenrich.data import read_csv_source, source_path
from laenrich.io import read_table, validate_analysis_mode
from laenrich.plotting import save_figure
from laenrich.prediction import compare_prediction_to_null, permutation_accuracy_null, prediction_accuracy
paths = load_project_paths(PROJECT_ROOT / "config" / "project.yaml")
mode = validate_analysis_mode(RUN_MODE)
checkpoint_dir = paths.checkpoint_reference_dir / "prediction" if mode == "checkpoint" else paths.checkpoint_generated_dir / "prediction"
figure_data_dir = paths.result_generated_dir / "figure_data" / "figure5"
figure_dir = paths.result_generated_dir / "figures"
supplementary_table_dir = paths.result_generated_dir / "supplementary_tables"
supplementary_dataset_dir = paths.result_generated_dir / "supplementary_datasets"
for directory in [figure_data_dir, figure_dir, supplementary_table_dir, supplementary_dataset_dir]:
    directory.mkdir(parents=True, exist_ok=True)
seed = int(RANDOM_SEED)
print({"mode": mode, "source_metadata": str(source_path(paths.source_dir, "gxe_metadata")), "checkpoint_dir": str(checkpoint_dir)})


## 1. GxE phenotype input and accession matching

The analysis-ready GxE metadata are matched to the genotype panel through LIB identifiers. Exactly 176 accessions with the required average phenotypes and plasticity traits are retained.


## 1. GxE phenotype input 与样本映射

通过 LIB identifiers 将 analysis-ready GxE metadata 与 genotype panel 匹配，最终保留恰好 176 个具有所需 average phenotypes 和 plasticity traits 的材料。


In [ ]:
phenotypes = read_csv_source(
    source_path(paths.source_dir, "gxe_metadata"),
    required_columns=["LIB", "Grain_yield_avg", "Grain_yield_plast", "Flowering_time_avg",
                      "Flowering_time_plast", "Plant_height_avg", "Plant_height_plast"],
).set_index("LIB", verify_integrity=True)
assert len(phenotypes) == int(EXPECTED_GXE_ACCESSION_COUNT)
phenotypes.head()


## 2. Analysis-ready average phenotypes and plasticity

GxE_metadata.csv already contains accession-level average phenotypes and plasticity values. This notebook standardizes those prepared traits and does not reconstruct WW/WS or site-year observations. Source column names are retained in the exported provenance table.


## 2. Analysis-ready average phenotypes 与 plasticity

GxE_metadata.csv 已包含 accession-level average phenotypes 和 plasticity values。本 notebook 对这些准备好的 traits 进行标准化，不重建 WW/WS 或 site-year observations。导出的 provenance table 保留 source column names。


In [ ]:
trait_columns = {
    "grain_yield_average": "Grain_yield_avg",
    "grain_yield_plasticity": "Grain_yield_plast",
    "flowering_time_average": "Flowering_time_avg",
    "flowering_time_plasticity": "Flowering_time_plast",
    "plant_height_average": "Plant_height_avg",
    "plant_height_plasticity": "Plant_height_plast",
}
derived_long = (
    phenotypes.loc[:, list(trait_columns.values())]
    .rename(columns={value: key for key, value in trait_columns.items()})
    .rename_axis("canonical_accession_id").reset_index()
    .melt(id_vars="canonical_accession_id", var_name="trait", value_name="phenotype")
)
derived_long["phenotype_z"] = derived_long.groupby("trait")["phenotype"].transform(
    lambda values: (values - values.mean()) / values.std(ddof=1)
)
grain_yield_plasticity = phenotypes["Grain_yield_plast"].rename("grain_yield_plasticity")
derived_long.head()


## 3. Context-specific GSL genetic scores

LDAK REML is fitted to GSL in the georeferenced landraces. BLUP effects are projected into the Brazil accessions for ACR, UMR, proximal-TE, matched-control, regulatory-union, and genome-wide variant sets.


## 3. Context-specific GSL genetic scores

在 georeferenced landraces 中使用 LDAK REML 拟合 GSL，并将 BLUP effects 投射到 Brazil accessions，覆盖 ACR、UMR、proximal-TE、matched-control、regulatory-union 和 genome-wide variant sets。


In [ ]:
prediction_scores = read_table(
    checkpoint_dir / "gsl_prediction_scores.h5",
    ["set_id", "context", "set_type", "replicate", "canonical_accession_id", "predicted_gsl"],
)
assert prediction_scores["canonical_accession_id"].isin(grain_yield_plasticity.index).all()
required_contexts = {"acr", "umr", "proximal_te", "regulatory_union", "genome"}
assert required_contexts.issubset(set(prediction_scores["context"]))
prediction_scores.groupby(["context", "set_type"])["canonical_accession_id"].nunique()


## 4. Prediction accuracy and empirical nulls

Accuracy is Pearson correlation between predicted GSL score and grain-yield plasticity. Context predictions are compared with 1,000 matched-control variant sets; a fixed-seed phenotype permutation provides a separate label-exchange null.


## 4. 预测准确度与 empirical nulls

Accuracy 定义为 predicted GSL score 与 grain-yield plasticity 的 Pearson correlation。Context predictions 与 1,000 个 matched-control variant sets 比较；固定 seed 的 phenotype permutation 另提供 label-exchange null。


In [ ]:
accuracy_records = []
for set_id, group in prediction_scores.groupby("set_id", sort=True):
    row = group.iloc[0]
    predicted = group.set_index("canonical_accession_id")["predicted_gsl"]
    metrics = prediction_accuracy(predicted, grain_yield_plasticity)
    accuracy_records.append({"set_id": set_id, "context": row["context"], "set_type": row["set_type"], "replicate": row["replicate"], **metrics})
accuracy = pd.DataFrame.from_records(accuracy_records)
context_summary_records = []
for context, group in accuracy.groupby("context", sort=True):
    observed = group.loc[group["set_type"].eq("context"), "pearson_r"]
    controls = group.loc[group["set_type"].eq("matched_control"), "pearson_r"].to_numpy()
    if len(observed) == 1 and len(controls) > 0:
        context_summary_records.append({"context": context, "observed_r": float(observed.iloc[0]), **compare_prediction_to_null(float(observed.iloc[0]), controls)})
prediction_summary = pd.DataFrame.from_records(context_summary_records)
primary_scores = prediction_scores.loc[prediction_scores["set_type"].eq("context")].groupby("context", sort=True)
permutation_records = []
for context, group in primary_scores:
    predicted = group.set_index("canonical_accession_id")["predicted_gsl"]
    observed = prediction_accuracy(predicted, grain_yield_plasticity)
    null = permutation_accuracy_null(predicted, grain_yield_plasticity, replicates=1000, seed=seed)
    permutation_records.append({"context": context, **observed, **compare_prediction_to_null(observed["pearson_r"], null)})
permutation_summary = pd.DataFrame.from_records(permutation_records)
prediction_summary


## 5. Figure 5 and exported data

Figure 5 shows observed accuracy for each functional context, matched-control distributions, and the genome-wide reference. Machine-readable phenotype and prediction tables retain P values and sample sizes.


## 5. Figure 5 与导出数据

Figure 5 展示各 functional context 的 observed accuracy、matched-control distributions 和 genome-wide reference。机器可读的 phenotype 与 prediction tables 保留 P values 和样本数。


In [ ]:
plot_contexts = sorted(set(accuracy["context"]))
figure, axis = plt.subplots(figsize=(8, 4))
for position, context in enumerate(plot_contexts):
    group = accuracy.loc[accuracy["context"].eq(context)]
    controls = group.loc[group["set_type"].eq("matched_control"), "pearson_r"]
    if len(controls):
        axis.boxplot(controls, positions=[position], widths=0.5, showfliers=False)
    observed = group.loc[group["set_type"].eq("context"), "pearson_r"]
    if len(observed):
        axis.scatter([position], [observed.iloc[0]], color="#b22222", zorder=3)
axis.axhline(0, color="black", linewidth=0.8)
axis.set_xticks(range(len(plot_contexts)), plot_contexts, rotation=35, ha="right")
axis.set_ylabel("Prediction accuracy (Pearson r)")
figure.tight_layout()
save_figure(figure, figure_dir / "figure5.pdf")
plt.close(figure)
accuracy.to_csv(supplementary_table_dir / "table_s5_prediction_accuracy.tsv", sep="\t", index=False)
prediction_summary.to_csv(figure_data_dir / "prediction_matched_null_summary.tsv", sep="\t", index=False)
permutation_summary.to_csv(figure_data_dir / "prediction_permutation_summary.tsv", sep="\t", index=False)
derived_long.to_csv(supplementary_dataset_dir / "dataset_s5_drought_phenotypes.tsv", sep="\t", index=False)
prediction_scores.to_hdf(figure_data_dir / "prediction_plot_data.h5", key="df", mode="w", format="table", index=False, complevel=9, complib="blosc:zstd")


## Manual recomputation command

LDAK REML, BLUP estimation, and projection into the Brazil panel are performed outside the notebook:

[SCHEDULER JOB]

```bash
python src/run_ldak_prediction.py --vcf <combined-panel.vcf.gz> --phenotype <phenotype.tsv> --sample-column <sample-column> --phenotype-column <gsl-column> --output <prediction-scores.tsv> --work-dir <temporary-work-directory> --plink <plink-executable> --ldak <ldak-executable>
```

The score table must follow the same schema as the published prediction checkpoint before this notebook is continued.


## 手动重算命令

LDAK REML、BLUP estimation 和向 Brazil panel 的 projection 在 notebook 外执行：

[SCHEDULER JOB]

```bash
python src/run_ldak_prediction.py --vcf <combined-panel.vcf.gz> --phenotype <phenotype.tsv> --sample-column <sample-column> --phenotype-column <gsl-column> --output <prediction-scores.tsv> --work-dir <temporary-work-directory> --plink <plink-executable> --ldak <ldak-executable>
```

继续运行本 notebook 前，score table 必须与已发布的 prediction checkpoint 使用相同 schema。


## Saved outputs and next step

Generated files remain in the remote working directory. Inspect the reported row counts, identifiers, quality-control summaries, and figure panels before proceeding to the next numbered notebook. External-tool commands described above remain manual steps.


## 已保存输出与下一步

生成的文件保留在远程工作目录中。进入下一本编号 notebook 前，请检查报告的行数、标识符、quality-control summaries 和 figure panels。以上 external-tool commands 仍需人工执行。
